# 03 – Privacy & Governance Compliance Audit

**Role:** Governance Officer  
**Task Force:** NovaCred  

## 1. Executive Summary & Scope
As the Governance Task Force, our objective is to audit the NovaCred credit application ecosystem. While the Data Engineering and Data Science teams focused on data quality and algorithmic fairness (`01-data-quality.ipynb` and `02-bias-analysis.ipynb`), this document provides a comprehensive **Regulatory Compliance Audit**.

This audit evaluates the system against the **General Data Protection Regulation (GDPR)** and the **EU Artificial Intelligence Act (AI Act)**. We will identify compliance gaps, demonstrate technical privacy safeguards, and outline a mandatory governance roadmap.

## 2. PII Inventory & Lawful Basis Mapping (GDPR Art. 6 & 9)

To govern data effectively, we must first map what we possess. Below is the programmatic PII inventory classifying the sensitivity and the lawful basis for processing the fields found in `raw_credit_applications.json`.

In [2]:
import pandas as pd

# Constructing the Data Governance Catalog
governance_catalog = pd.DataFrame({
    'Field_Name': ['full_name', 'email', 'ssn', 'ip_address', 'date_of_birth', 'zip_code', 'spending_behavior'],
    'Identifier_Type': ['Direct', 'Direct', 'Direct (Critical)', 'Quasi-Identifier', 'Quasi-Identifier', 'Quasi-Identifier', 'Behavioral/Sensitive'],
    'GDPR_Lawful_Basis': ['Art. 6(1)(b) Contract', 'Art. 6(1)(b) Contract', 'Art. 6(1)(b) Contract', 'None (Violation)', 'Art. 6(1)(b) Contract', 'None (Violation)', 'Art. 9 Explicit Consent Req.'],
    'Re_identification_Risk': ['Low', 'Low', 'High', 'Medium', 'High (if combined)', 'High (Proxy Risk)', 'Medium']
})

print("NovaCred Data Governance Catalog")
display(governance_catalog)

NovaCred Data Governance Catalog


,Field_Name,Identifier_Type,GDPR_Lawful_Basis,Re_identification_Risk
0,full_name,Direct,Art. 6(1)(b) Contract,Low
1,email,Direct,Art. 6(1)(b) Contract,Low
2,ssn,Direct (Critical),Art. 6(1)(b) Contract,High
3,ip_address,Quasi-Identifier,None (Violation),Medium
4,date_of_birth,Quasi-Identifier,Art. 6(1)(b) Contract,High (if combined)
5,zip_code,Quasi-Identifier,None (Violation),High (Proxy Risk)
6,spending_behavior,Behavioral/Sensitive,Art. 9 Explicit Consent Req.,Medium


### 2.1 Critical Violations Identified
- **Article 5(1)(c) - Data Minimization:** The collection of `ip_address` provides no legitimate value for establishing creditworthiness. Its presence is an unnecessary exposure of personal metadata.
- **Article 9 - Special Categories of Data:** The `spending_behavior` array contains raw transaction tags. Alarmingly, application `app_200` includes the category **"Alcohol"**. Tracking such data infers health status and potential addictions. Processing this data without explicit, informed consent is a severe breach of GDPR.
- **Fairness and Proxy Discrimination:** As confirmed in `02-bias-analysis.ipynb`, `zip_code` serves as a geographic proxy for protected attributes. Retaining it violates the ethical guidelines of the AI Act and perpetuates historical biases.

## 3. Technical Safeguards: Privacy by Design (Art. 25 GDPR)

To comply with the principle of **Privacy by Design**, we must engineer privacy into the ingestion pipeline. We distinguish between:
1. **Anonymization:** Irreversibly destroying the link to the individual (data is no longer subject to GDPR).
2. **Pseudonymization:** Replacing direct identifiers with a surrogate value (e.g., hashing). The data remains subject to GDPR but security is vastly improved.

### Demo 3.1: Data Minimization & Pseudonymization Pipeline

In [3]:
import hashlib
import numpy as np

# Simulating a batch of raw ingested data
raw_batch = pd.DataFrame({
    'app_id': ['app_200', 'app_306', 'app_163'],
    'full_name': ['Jerry Smith', 'Anna White', 'Edward White'],
    'ssn': ['596-64-4340', '757-27-8131', '684-21-9902'],
    'ip_address': ['192.168.48.155', '10.26.176.56', '172.16.254.1']
})

def secure_ingestion_pipeline(df):
    """Applies automated privacy controls to raw data."""
    df_secure = df.copy()
    
    # 1. Data Minimization: Drop non-essential fields
    df_secure = df_secure.drop(columns=['ip_address', 'full_name'])
    
    # 2. Pseudonymization: Hash the SSN (Using SHA-256)
    # In production, a secure 'salt' managed by a Key Management System (KMS) is required.
    def hash_identifier(val):
        if pd.isna(val): return np.nan
        return hashlib.sha256(str(val).encode('utf-8')).hexdigest()[:16] # Truncated for readability
        
    df_secure['ssn_token'] = df_secure['ssn'].apply(hash_identifier)
    df_secure = df_secure.drop(columns=['ssn']) # Destroy the plain-text column
    
    return df_secure

secured_data = secure_ingestion_pipeline(raw_batch)
print("Analytics Zone: Secured Data")
display(secured_data)

Analytics Zone: Secured Data


,app_id,ssn_token
0,app_200,2caf30528c21a10e
1,app_306,618befc68f8bfb7b
2,app_163,338011da6352b21b


### Demo 3.2: Right to Erasure (Art. 17 GDPR)
Under GDPR, applicants have the "Right to be Forgotten". The database must support the hard deletion of PII upon request.

In [4]:
def execute_right_to_erasure(df, target_app_id):
    """Simulates the erasure of a user's record from the active database."""
    print(f"\n[AUDIT LOG] Received Art. 17 Erasure Request for: {target_app_id}")
    
    # Verify existence
    if target_app_id in df['app_id'].values:
        # Perform hard delete
        df_deleted = df[df['app_id'] != target_app_id].reset_index(drop=True)
        print(f"[SUCCESS] Record {target_app_id} permanently erased.")
        return df_deleted
    else:
        print(f"[WARNING] Record {target_app_id} not found.")
        return df

# Simulating User 'app_200' exercising their Right to Erasure
secured_data = execute_right_to_erasure(secured_data, 'app_200')
display(secured_data)


[AUDIT LOG] Received Art. 17 Erasure Request for: app_200
[SUCCESS] Record app_200 permanently erased.


,app_id,ssn_token
0,app_306,618befc68f8bfb7b
1,app_163,338011da6352b21b


## 4. EU AI Act Classification & Actionable Roadmap

The NovaCred Machine Learning model evaluates the creditworthiness of natural persons. According to **Annex III, Point 5(b)** of the **EU AI Act**, this automatically classifies it as a **High-Risk AI System**. 

To avoid severe regulatory penalties and ensure ethical AI deployment, we propose the following mandatory Governance Roadmap:

### 4.1 Process & Policy Controls
1. **Human Oversight (AI Act Art. 14):** Automated rejections based purely on `algorithm_risk_score` are currently unlawful. We must institute a **Credit Review Committee**. This "Human-in-the-Loop" (HITL) system ensures that applicants can appeal automated rejections and have their profiles reviewed by a human officer.
2. **Transparency (AI Act Art. 13):** The system must generate human-readable explanations. Rejecting an applicant with `rejection_reason: "algorithm_risk_score"` provides no interpretability and violates transparency mandates.
3. **Data Retention Policy (GDPR Art. 5(1)(e)):** Implement an automated data lifecycle rule. The records of rejected applicants must be purged after 5 years (the standard period for financial audit limitations).

### 4.2 Technical & Algorithmic Controls
1. **Bias-Gate Deployment (AI Act Art. 10):** Integrating the findings from the Data Science team, we must establish an automated CI/CD pipeline check. If the **Disparate Impact Ratio (DIR)** for any protected attribute (e.g., gender, age) falls below the 0.8 threshold, the model deployment must be automatically blocked.
2. **Feature Pruning:** Remove `zip_code` from the training features to eliminate geographic proxy discrimination.
3. **Audit Trails (AI Act Art. 12):** Implement logging mechanisms that record exactly which version of the ML model made a decision, the input features used, and the confidence score. This trail must be immutable and accessible for regulatory inspections.